# rabi log bayes histograms
See the repository README for the paper configurations and cached-data reproduction commands.


In [ ]:
from __future__ import annotations

import hashlib
import importlib
import json
import math
import os
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "dm_search").exists():
            return candidate
    raise RuntimeError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import dm_search.rabi as _rabi_module  # noqa: E402
_rabi_module = importlib.reload(_rabi_module)

from dm_search.rabi import (  # noqa: E402
    RabiExperimentConfig,
    RabiParticleSpec,
    RabiReadoutConfig,
    build_rabi_particle_grid,
    longitudinal_relaxation_factor,
    rabi_probability_one,
    sample_rabi_log_bayes_factors,
    sample_rabi_diagnostics_at_checkpoints,
    transverse_relaxation_factor,
    with_true_phase_amplitude,
)
from dm_search.statistics import null_threshold  # noqa: E402
from dm_search.rabi_prefix_cache import (  # noqa: E402
    RABI_PREFIX_DETECTOR_PROFILES,
    assert_matching_raw_setup,
    build_rabi_prefix_setup,
    rabi_prefix_cache_path,
)

# Shared manuscript typography and frameless legends.
plt.style.use(REPO_ROOT / "plot_final" / "rabi_paper.mplstyle")


In [ ]:
# Physics and test setup.  These match rabi_posterior_learning unless changed here.
ALPHA = 0.05
SENSING_TIME = 1.0
T1_TIME = SENSING_TIME
T2_TIME = SENSING_TIME
DEAD_TIME = 0.0
TRUE_DETUNING = 0.0
DETECTOR_PROFILE = os.environ.get("RABI_HIST_DETECTOR_PROFILE", "current")
if DETECTOR_PROFILE not in RABI_PREFIX_DETECTOR_PROFILES:
    raise ValueError(
        f"Unknown RABI_HIST_DETECTOR_PROFILE={DETECTOR_PROFILE!r}; "
        f"choose one of {sorted(RABI_PREFIX_DETECTOR_PROFILES)}"
    )
DETECTOR_DEFAULTS = RABI_PREFIX_DETECTOR_PROFILES[DETECTOR_PROFILE]
CONTRAST = float(os.environ.get("RABI_HIST_CONTRAST", DETECTOR_DEFAULTS["contrast"]))
READOUT_ERROR = float(os.environ.get("RABI_HIST_READOUT_ERROR", DETECTOR_DEFAULTS["readout_error"]))
INITIAL_H1_PROBABILITY = 0.5

# Monte Carlo controls.  The paper configuration is the default; override for quick checks.
RUN_PRESET = os.environ.get("RABI_HIST_PRESET", "final")
PRESETS = {
    "quick": {"n_shots": 128, "amplitude_count": 18, "drive_phase_count": 36, "candidate_angle_count": 16, "candidate_polar_count": 5, "n_null": 128, "n_signal": 128, "batch_size": 512, "bin_count": 44, "phase_average_count": 12, "readout_space": "sphere", "readout_optimization": "continuous", "readout_objective": "periodic_hybrid", "periodic_hybrid_period": 1, "readout_switch_shot": 128, "blend_lambda_max": 1.0, "blend_r_start": 0.2, "blend_r_stop": 0.9},
    "draft": {"n_shots": 512, "amplitude_count": 24, "drive_phase_count": 48, "candidate_angle_count": 16, "candidate_polar_count": 5, "n_null": 2_000, "n_signal": 2_000, "batch_size": 512, "bin_count": 70, "phase_average_count": 24, "readout_space": "sphere", "readout_optimization": "continuous", "readout_objective": "periodic_hybrid", "periodic_hybrid_period": 1, "readout_switch_shot": 512, "blend_lambda_max": 1.0, "blend_r_start": 0.2, "blend_r_stop": 0.9},
    "final": {"n_shots": 1024, "amplitude_count": 48, "drive_phase_count": 96, "candidate_angle_count": 16, "candidate_polar_count": 7, "n_null": 1_000, "n_signal": 1_000, "batch_size": 512, "bin_count": 65, "phase_average_count": 96, "readout_space": "sphere", "readout_optimization": "continuous", "readout_objective": "periodic_hybrid", "periodic_hybrid_period": 1, "readout_switch_shot": 1024, "blend_lambda_max": 1.0, "blend_r_start": 0.2, "blend_r_stop": 0.9},
    "final_10k": {"n_shots": 1024, "amplitude_count": 48, "drive_phase_count": 96, "candidate_angle_count": 16, "candidate_polar_count": 7, "n_null": 10_000, "n_signal": 10_000, "batch_size": 512, "bin_count": 90, "phase_average_count": 96, "readout_space": "sphere", "readout_optimization": "continuous", "readout_objective": "periodic_hybrid", "periodic_hybrid_period": 1, "readout_switch_shot": 1024, "blend_lambda_max": 1.0, "blend_r_start": 0.2, "blend_r_stop": 0.9},
    "long": {"n_shots": 1024, "amplitude_count": 48, "drive_phase_count": 96, "candidate_angle_count": 16, "candidate_polar_count": 7, "n_null": 50_000, "n_signal": 50_000, "batch_size": 512, "bin_count": 110, "phase_average_count": 192, "readout_space": "sphere", "readout_optimization": "continuous", "readout_objective": "periodic_hybrid", "periodic_hybrid_period": 1, "readout_switch_shot": 1024, "blend_lambda_max": 1.0, "blend_r_start": 0.2, "blend_r_stop": 0.9},
}
if RUN_PRESET not in PRESETS:
    raise ValueError(f"Unknown RUN_PRESET={RUN_PRESET!r}; choose one of {sorted(PRESETS)}")

N_SHOTS = int(os.environ.get("RABI_HIST_N_SHOTS", PRESETS[RUN_PRESET]["n_shots"]))
HIST_PHASE = float(os.environ.get("RABI_HIST_PHI_R", "0.19"))
N_NULL = int(os.environ.get("RABI_HIST_N_NULL", PRESETS[RUN_PRESET]["n_null"]))
N_SIGNAL = int(os.environ.get("RABI_HIST_N_SIGNAL", PRESETS[RUN_PRESET]["n_signal"]))
BATCH_SIZE = int(os.environ.get("RABI_HIST_BATCH_SIZE", PRESETS[RUN_PRESET]["batch_size"]))
BIN_COUNT = int(os.environ.get("RABI_HIST_BIN_COUNT", PRESETS[RUN_PRESET]["bin_count"]))
SAMPLER_METHOD = "numba"
SEED_BASE = 73_000
PHASE_AVERAGE_MODE = os.environ.get("RABI_HIST_PHASE_MODE", "random_per_experiment")
PHASE_AVERAGE_COUNT = int(os.environ.get("RABI_HIST_PHASE_COUNT", PRESETS[RUN_PRESET]["phase_average_count"]))
WARMUP_NUMBA = os.environ.get("RABI_HIST_WARMUP_NUMBA", "1") not in {"0", "false", "False"}

# Cache and output controls.
USE_CACHE = os.environ.get("RABI_HIST_USE_CACHE", "1") not in {"0", "false", "False"}
USE_PREFIX_RAW_CACHE = os.environ.get("RABI_HIST_USE_PREFIX_RAW_CACHE", "1") not in {"0", "false", "False"}
PREFIX_RAW_CACHE_OVERRIDE = os.environ.get("RABI_HIST_PREFIX_RAW_CACHE", "").strip()
# H1 logB + A49 diagnostics: first final run takes about 40-50 minutes.
H1_KL_DIAGNOSTICS = os.environ.get("RABI_HIST_H1_KL", "1") not in {"0", "false", "False"}
FORCE_RECOMPUTE = os.environ.get("RABI_HIST_FORCE_RECOMPUTE", "0") in {"1", "true", "True"}
CACHE_DIR = REPO_ROOT / "plot_final" / "cache"
FIGURE_DIR = REPO_ROOT / "plot_final" / "figure"
OUTPUT_BASENAME = "rabi_log_bayes_histograms"
CACHE_VERSION = "particle-samplers-v8-current-detector-profile"
SAVE_FIGURES = True

STRATEGIES = {
    "fixed_z": {"label": r"Fixed $z$", "panel": "a", "color": "#536B78"},
    "fixed_x": {"label": r"Fixed transverse $x$", "panel": "b", "color": "#8F6A35"},
    "two_axis": {"label": r"Fixed transverse $x/y$", "panel": "c", "color": "#4E7F59"},
    "adaptive": {"label": r"Adaptive $L=1$ (IG)", "panel": "d", "color": "#9A4D55"},
}

# H1 model particles.  These are inference/prior particles, not the true signal values.
# Use PARTICLE_MODE="grid" for the Cartesian grid, "random" for iid prior particles,
# "scheduled" for deterministic Delta refinement, "smc" for raw resampling+MCMC moves,
# "smc_refined" for SMC-guided actions with deterministic refined evidence replay,
# or "multires_grid" for deterministic Delta-grid refinement as shots accumulate.
# If *_MIN == *_MAX, that parameter is treated as known and is not sampled.
# In grid mode, *_COUNT = 0 is allowed only for such known min=max parameters.
PARTICLE_MODE = os.environ.get("RABI_HIST_PARTICLE_MODE", "grid")
PARTICLE_PHASE_AMPLITUDE_MIN = float(os.environ.get("RABI_HIST_AMPLITUDE_MIN", "0.02"))
PARTICLE_PHASE_AMPLITUDE_MAX = float(os.environ.get("RABI_HIST_AMPLITUDE_MAX", "1.2"))
PARTICLE_PHASE_AMPLITUDE_COUNT = int(os.environ.get("RABI_HIST_AMPLITUDE_COUNT", PRESETS[RUN_PRESET]["amplitude_count"]))
PARTICLE_PHASE_AMPLITUDE_PRIOR = os.environ.get("RABI_HIST_AMPLITUDE_PRIOR", "log_uniform")
PARTICLE_DETUNING_MIN = 0.
PARTICLE_DETUNING_MAX = 0.
PARTICLE_DETUNING_COUNT = 0
PARTICLE_DRIVE_PHASE_COUNT = int(os.environ.get("RABI_HIST_DRIVE_PHASE_COUNT", PRESETS[RUN_PRESET]["drive_phase_count"]))
PARTICLE_RANDOM_COUNT = int(os.environ.get("RABI_HIST_RANDOM_PARTICLES", "512"))
PARTICLE_RANDOM_SEED = int(os.environ.get("RABI_HIST_RANDOM_SEED", "24681357"))
SCHEDULED_INITIAL_DETUNING_COUNT = int(os.environ.get("RABI_HIST_SCHEDULED_INITIAL_DETUNING_COUNT", "1"))
SCHEDULED_REFINE_PHASE_TOLERANCE = float(os.environ.get("RABI_HIST_SCHEDULED_REFINE_PHASE_TOLERANCE", "1.0"))
SCHEDULED_PRUNE_RELATIVE_THRESHOLD = float(os.environ.get("RABI_HIST_SCHEDULED_PRUNE_RELATIVE_THRESHOLD", "1e-8"))
SCHEDULED_PRUNE_CUMULATIVE_MASS = float(os.environ.get("RABI_HIST_SCHEDULED_PRUNE_CUMULATIVE_MASS", "0.999999"))
SMC_RESAMPLE_ESS_FRACTION = float(os.environ.get("RABI_HIST_SMC_RESAMPLE_ESS_FRACTION", "0.5"))
SMC_MOVE_STEPS = int(os.environ.get("RABI_HIST_SMC_MOVE_STEPS", "1"))
SMC_LOG_PHASE_AMPLITUDE_STEP = float(os.environ.get("RABI_HIST_SMC_LOG_PHASE_AMPLITUDE_STEP", "0.15"))
SMC_DETUNING_STEP = float(os.environ.get("RABI_HIST_SMC_DETUNING_STEP", "1e-3"))
SMC_DRIVE_PHASE_STEP = float(os.environ.get("RABI_HIST_SMC_DRIVE_PHASE_STEP", "0.35"))
SMC_REFINE_DETUNING_CENTER_COUNT = int(os.environ.get("RABI_HIST_SMC_REFINE_DETUNING_CENTER_COUNT", "3"))
SMC_REFINE_DETUNING_LOCAL_COUNT = int(os.environ.get("RABI_HIST_SMC_REFINE_DETUNING_LOCAL_COUNT", "21"))
SMC_REFINE_DETUNING_COARSE_COUNT = int(os.environ.get("RABI_HIST_SMC_REFINE_DETUNING_COARSE_COUNT", "9"))
SMC_REFINE_DETUNING_WIDTH = float(os.environ.get("RABI_HIST_SMC_REFINE_DETUNING_WIDTH", "0.0"))
MULTIRES_INITIAL_DETUNING_COUNT = int(os.environ.get("RABI_HIST_MULTIRES_INITIAL_DETUNING_COUNT", "5"))
MULTIRES_INITIAL_SHOTS = int(os.environ.get("RABI_HIST_MULTIRES_INITIAL_SHOTS", "128"))
MULTIRES_REFINE_PHASE_TOLERANCE = float(os.environ.get("RABI_HIST_MULTIRES_REFINE_PHASE_TOLERANCE", "1.0"))
MULTIRES_FINAL_REPLAY = os.environ.get("RABI_HIST_MULTIRES_FINAL_REPLAY", "1") not in {"0", "false", "False"}

BASE_PARTICLES = RabiParticleSpec(
    particle_mode=PARTICLE_MODE,
    phase_amplitude_min=PARTICLE_PHASE_AMPLITUDE_MIN,
    phase_amplitude_max=PARTICLE_PHASE_AMPLITUDE_MAX,
    phase_amplitude_count=PARTICLE_PHASE_AMPLITUDE_COUNT,
    phase_amplitude_prior=PARTICLE_PHASE_AMPLITUDE_PRIOR,
    detuning_min=PARTICLE_DETUNING_MIN,
    detuning_max=PARTICLE_DETUNING_MAX,
    detuning_count=PARTICLE_DETUNING_COUNT,
    drive_phase_count=PARTICLE_DRIVE_PHASE_COUNT,
    random_particle_count=PARTICLE_RANDOM_COUNT,
    random_seed=PARTICLE_RANDOM_SEED,
    scheduled_initial_detuning_count=SCHEDULED_INITIAL_DETUNING_COUNT,
    scheduled_refine_phase_tolerance=SCHEDULED_REFINE_PHASE_TOLERANCE,
    scheduled_prune_relative_threshold=SCHEDULED_PRUNE_RELATIVE_THRESHOLD,
    scheduled_prune_cumulative_mass=SCHEDULED_PRUNE_CUMULATIVE_MASS,
    smc_resample_ess_fraction=SMC_RESAMPLE_ESS_FRACTION,
    smc_move_steps=SMC_MOVE_STEPS,
    smc_log_phase_amplitude_step=SMC_LOG_PHASE_AMPLITUDE_STEP,
    smc_detuning_step=SMC_DETUNING_STEP,
    smc_drive_phase_step=SMC_DRIVE_PHASE_STEP,
    smc_refine_detuning_center_count=SMC_REFINE_DETUNING_CENTER_COUNT,
    smc_refine_detuning_local_count=SMC_REFINE_DETUNING_LOCAL_COUNT,
    smc_refine_detuning_coarse_count=SMC_REFINE_DETUNING_COARSE_COUNT,
    smc_refine_detuning_width=SMC_REFINE_DETUNING_WIDTH,
    multires_initial_detuning_count=MULTIRES_INITIAL_DETUNING_COUNT,
    multires_initial_shots=MULTIRES_INITIAL_SHOTS,
    multires_refine_phase_tolerance=MULTIRES_REFINE_PHASE_TOLERANCE,
    multires_final_replay=MULTIRES_FINAL_REPLAY,
)
READOUT_OPTIMIZATION = os.environ.get("RABI_HIST_READOUT_OPTIMIZATION", PRESETS[RUN_PRESET]["readout_optimization"])
if READOUT_OPTIMIZATION not in {"grid", "continuous"}:
    raise ValueError("RABI_HIST_READOUT_OPTIMIZATION must be grid or continuous")
READOUT_SPACE = os.environ.get("RABI_HIST_READOUT_SPACE", PRESETS[RUN_PRESET].get("readout_space", "sphere"))
if READOUT_SPACE not in {"transverse", "sphere"}:
    raise ValueError("RABI_HIST_READOUT_SPACE must be transverse or sphere")
READOUT_OBJECTIVE = os.environ.get("RABI_HIST_READOUT_OBJECTIVE", PRESETS[RUN_PRESET]["readout_objective"])
if READOUT_OBJECTIVE not in {"information_gain", "bayes_drift", "hybrid", "blended", "periodic_hybrid", "periodic_bayes_hybrid"}:
    raise ValueError("RABI_HIST_READOUT_OBJECTIVE must be information_gain, bayes_drift, hybrid, blended, periodic_hybrid, or periodic_bayes_hybrid")
if READOUT_SPACE == "transverse" and READOUT_OBJECTIVE in {"periodic_hybrid", "periodic_bayes_hybrid"}:
    raise ValueError("periodic_hybrid objectives currently require RABI_HIST_READOUT_SPACE=sphere")
READOUT_SWITCH_SHOT = int(os.environ.get("RABI_HIST_READOUT_SWITCH_SHOT", PRESETS[RUN_PRESET].get("readout_switch_shot", 0)))
if READOUT_SWITCH_SHOT < 0:
    raise ValueError("RABI_HIST_READOUT_SWITCH_SHOT must be non-negative")
BLEND_LAMBDA_MAX = float(os.environ.get("RABI_HIST_BLEND_LAMBDA_MAX", PRESETS[RUN_PRESET].get("blend_lambda_max", 1.0)))
BLEND_R_START = float(os.environ.get("RABI_HIST_BLEND_R_START", PRESETS[RUN_PRESET].get("blend_r_start", 0.2)))
BLEND_R_STOP = float(os.environ.get("RABI_HIST_BLEND_R_STOP", PRESETS[RUN_PRESET].get("blend_r_stop", 0.9)))
if BLEND_LAMBDA_MAX < 0.0 or not (0.0 <= BLEND_R_START < BLEND_R_STOP <= 1.0):
    raise ValueError("blend parameters must satisfy lambda_max >= 0 and 0 <= R_start < R_stop <= 1")
candidate_count_text = os.environ.get("RABI_HIST_CANDIDATE_COUNT")
if READOUT_OPTIMIZATION == "grid" and candidate_count_text is None:
    raise ValueError("RABI_HIST_CANDIDATE_COUNT must be set when RABI_HIST_READOUT_OPTIMIZATION=grid")
CANDIDATE_ANGLE_COUNT = int(candidate_count_text) if candidate_count_text is not None else int(PRESETS[RUN_PRESET].get("candidate_angle_count", 16))
if CANDIDATE_ANGLE_COUNT <= 0:
    raise ValueError("RABI_HIST_CANDIDATE_COUNT must be positive")
READOUT_AZIMUTH_PERIOD = math.pi if READOUT_SPACE == "sphere" else 2.0 * math.pi
CANDIDATE_ANGLES = tuple(np.linspace(0.0, READOUT_AZIMUTH_PERIOD, CANDIDATE_ANGLE_COUNT, endpoint=False))
CANDIDATE_POLAR_COUNT = int(os.environ.get("RABI_HIST_CANDIDATE_POLAR_COUNT", PRESETS[RUN_PRESET].get("candidate_polar_count", 5)))
if CANDIDATE_POLAR_COUNT <= 0:
    raise ValueError("RABI_HIST_CANDIDATE_POLAR_COUNT must be positive")
CANDIDATE_POLARS = tuple(np.linspace(0.0, math.pi, CANDIDATE_POLAR_COUNT)) if READOUT_SPACE == "sphere" else (0.5 * math.pi,)
PERIODIC_HYBRID_PERIOD = int(os.environ.get("RABI_HIST_PERIODIC_HYBRID_PERIOD", PRESETS[RUN_PRESET].get("periodic_hybrid_period", 8)))
if PERIODIC_HYBRID_PERIOD <= 0:
    raise ValueError("RABI_HIST_PERIODIC_HYBRID_PERIOD must be positive")
if READOUT_OBJECTIVE == "periodic_hybrid":
    ADAPTIVE_SAMPLER_STRATEGY = f"periodic_hybrid_sphere_continuous_{PERIODIC_HYBRID_PERIOD}"
elif READOUT_OBJECTIVE == "periodic_bayes_hybrid":
    ADAPTIVE_SAMPLER_STRATEGY = f"periodic_bayes_hybrid_sphere_continuous_{PERIODIC_HYBRID_PERIOD}"
elif READOUT_OBJECTIVE == "blended":
    blend_suffix = f"{BLEND_LAMBDA_MAX:g}_{BLEND_R_START:g}_{BLEND_R_STOP:g}"
    ADAPTIVE_SAMPLER_STRATEGY = f"adaptive_blended_continuous_{blend_suffix}" if READOUT_OPTIMIZATION == "continuous" else f"adaptive_blended_{blend_suffix}"
elif READOUT_OBJECTIVE == "hybrid":
    ADAPTIVE_SAMPLER_STRATEGY = f"adaptive_hybrid_continuous_{READOUT_SWITCH_SHOT}" if READOUT_OPTIMIZATION == "continuous" else f"adaptive_hybrid_{READOUT_SWITCH_SHOT}"
elif READOUT_OBJECTIVE == "bayes_drift":
    ADAPTIVE_SAMPLER_STRATEGY = "adaptive_bayes_continuous" if READOUT_OPTIMIZATION == "continuous" else "adaptive_bayes"
else:
    ADAPTIVE_SAMPLER_STRATEGY = "adaptive_continuous" if READOUT_OPTIMIZATION == "continuous" else "adaptive"

if PHASE_AVERAGE_MODE == "uniform_grid":
    TRUE_DRIVE_PHASES = tuple(np.linspace(0.0, 2.0 * np.pi, PHASE_AVERAGE_COUNT, endpoint=False))
elif PHASE_AVERAGE_MODE == "uniform_midpoint":
    phase_step = 2.0 * np.pi / PHASE_AVERAGE_COUNT
    TRUE_DRIVE_PHASES = tuple(np.linspace(0.0, 2.0 * np.pi, PHASE_AVERAGE_COUNT, endpoint=False) + 0.5 * phase_step)
elif PHASE_AVERAGE_MODE == "random_grid":
    phase_rng = np.random.default_rng(SEED_BASE + 91_337)
    TRUE_DRIVE_PHASES = tuple(np.sort(phase_rng.uniform(0.0, 2.0 * np.pi, PHASE_AVERAGE_COUNT)))
elif PHASE_AVERAGE_MODE == "random_per_experiment":
    TRUE_DRIVE_PHASES = tuple()
else:
    raise ValueError("PHASE_AVERAGE_MODE must be 'uniform_grid', 'uniform_midpoint', 'random_grid', or 'random_per_experiment'")


In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

prefix_raw_setup = build_rabi_prefix_setup(
    "final",
    environ={
        "RABI_PREFIX_DETECTOR_PROFILE": DETECTOR_PROFILE,
        "RABI_PREFIX_CONTRAST": str(CONTRAST),
        "RABI_PREFIX_READOUT_ERROR": str(READOUT_ERROR),
        "RABI_PREFIX_ADAPTIVE_PERIODS": "1,2",
    },
)
PREFIX_RAW_CACHE_PATH = (
    Path(PREFIX_RAW_CACHE_OVERRIDE).expanduser().resolve()
    if PREFIX_RAW_CACHE_OVERRIDE
    else rabi_prefix_cache_path(CACHE_DIR, "final", prefix_raw_setup.raw_setup)
)

# Reuse the published figure's source if the current scan grid has a new hash.
# The loader below still checks the model, prior, policy, and requested slice.
if not PREFIX_RAW_CACHE_OVERRIDE and not PREFIX_RAW_CACHE_PATH.exists():
    provenance_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_setup.json"
    if provenance_path.exists():
        provenance = json.loads(provenance_path.read_text())
        if provenance.get("h0_sample_source", provenance.get("sample_source")) == "rabi_prefix_raw_data":
            PREFIX_RAW_CACHE_PATH = CACHE_DIR / Path(provenance["prefix_raw_cache_path"]).name

base_config = RabiExperimentConfig(
    n_shots=N_SHOTS,
    sensing_time=SENSING_TIME,
    dead_time=DEAD_TIME,
    true_phase_amplitude=HIST_PHASE,
    true_detuning=TRUE_DETUNING,
    true_drive_phase=0.0,
    initial_h1_probability=INITIAL_H1_PROBABILITY,
    readout=RabiReadoutConfig(contrast=CONTRAST, readout_error=READOUT_ERROR, t1=T1_TIME, t2=T2_TIME),
    particles=BASE_PARTICLES,
    candidate_angles=CANDIDATE_ANGLES,
    candidate_polars=CANDIDATE_POLARS,
    seed=SEED_BASE,
)

setup = {
    "run_preset": RUN_PRESET,
    "detector_profile": DETECTOR_PROFILE,
    "alpha": ALPHA,
    "n_shots": N_SHOTS,
    "hist_phase": HIST_PHASE,
    "n_null": N_NULL,
    "n_signal_phase_averaged": N_SIGNAL,
    "batch_size": BATCH_SIZE,
    "sampler_method": SAMPLER_METHOD,
    "cache_version": CACHE_VERSION,
    "use_cache": USE_CACHE,
    "use_prefix_raw_cache": USE_PREFIX_RAW_CACHE,
    "prefix_raw_cache_path": str(PREFIX_RAW_CACHE_PATH),
    "force_recompute": FORCE_RECOMPUTE,
    "true_detuning": TRUE_DETUNING,
    "true_drive_phase_note": "fixed_z is independent of true_drive_phase; transverse H1 samples follow phase_average_mode",
    "phase_average_mode": PHASE_AVERAGE_MODE,
    "phase_average_count": PHASE_AVERAGE_COUNT,
    "true_drive_phase_values": [float(x) for x in TRUE_DRIVE_PHASES],
    "sensing_time": SENSING_TIME,
    "t1_time": T1_TIME,
    "t2_time": T2_TIME,
    "eta2_transverse": transverse_relaxation_factor(SENSING_TIME, T2_TIME),
    "eta1_longitudinal": longitudinal_relaxation_factor(SENSING_TIME, T1_TIME, T2_TIME),
    "readout": {"contrast": CONTRAST, "readout_error": READOUT_ERROR},
    "particle_spec": {
        "particle_mode": PARTICLE_MODE,
        "phase_amplitude_min": PARTICLE_PHASE_AMPLITUDE_MIN,
        "phase_amplitude_max": PARTICLE_PHASE_AMPLITUDE_MAX,
        "phase_amplitude_count": PARTICLE_PHASE_AMPLITUDE_COUNT,
        "phase_amplitude_prior": PARTICLE_PHASE_AMPLITUDE_PRIOR,
        "detuning_min": PARTICLE_DETUNING_MIN,
        "detuning_max": PARTICLE_DETUNING_MAX,
        "detuning_count": PARTICLE_DETUNING_COUNT,
        "drive_phase_count": PARTICLE_DRIVE_PHASE_COUNT,
        "random_particle_count": PARTICLE_RANDOM_COUNT,
        "random_seed": PARTICLE_RANDOM_SEED,
        "scheduled_initial_detuning_count": SCHEDULED_INITIAL_DETUNING_COUNT,
        "scheduled_refine_phase_tolerance": SCHEDULED_REFINE_PHASE_TOLERANCE,
        "scheduled_prune_relative_threshold": SCHEDULED_PRUNE_RELATIVE_THRESHOLD,
        "scheduled_prune_cumulative_mass": SCHEDULED_PRUNE_CUMULATIVE_MASS,
        "smc_resample_ess_fraction": SMC_RESAMPLE_ESS_FRACTION,
        "smc_move_steps": SMC_MOVE_STEPS,
        "smc_log_phase_amplitude_step": SMC_LOG_PHASE_AMPLITUDE_STEP,
        "smc_detuning_step": SMC_DETUNING_STEP,
        "smc_drive_phase_step": SMC_DRIVE_PHASE_STEP,
        "smc_refine_detuning_center_count": SMC_REFINE_DETUNING_CENTER_COUNT,
        "smc_refine_detuning_local_count": SMC_REFINE_DETUNING_LOCAL_COUNT,
        "smc_refine_detuning_coarse_count": SMC_REFINE_DETUNING_COARSE_COUNT,
        "smc_refine_detuning_width": SMC_REFINE_DETUNING_WIDTH,
        "multires_initial_detuning_count": MULTIRES_INITIAL_DETUNING_COUNT,
        "multires_initial_shots": MULTIRES_INITIAL_SHOTS,
        "multires_refine_phase_tolerance": MULTIRES_REFINE_PHASE_TOLERANCE,
        "multires_final_replay": MULTIRES_FINAL_REPLAY,
    },
    "particle_shape": build_rabi_particle_grid(BASE_PARTICLES).shape,
    "particle_count": int(np.prod(build_rabi_particle_grid(BASE_PARTICLES).shape)),
    "readout_space": READOUT_SPACE,
    "readout_azimuth_period": READOUT_AZIMUTH_PERIOD,
    "readout_optimization": READOUT_OPTIMIZATION,
    "readout_objective": READOUT_OBJECTIVE,
    "periodic_hybrid_period": PERIODIC_HYBRID_PERIOD,
    "readout_switch_shot": READOUT_SWITCH_SHOT,
    "blend_lambda_max": BLEND_LAMBDA_MAX,
    "blend_r_start": BLEND_R_START,
    "blend_r_stop": BLEND_R_STOP,
    "adaptive_sampler_strategy": ADAPTIVE_SAMPLER_STRATEGY,
    "candidate_angle_count": len(CANDIDATE_ANGLES),
    "candidate_polar_count": len(CANDIDATE_POLARS),
    "strategies": {key: meta["label"] for key, meta in STRATEGIES.items()},
}
display(setup)


In [ ]:
def stable_hash(payload: dict) -> str:
    encoded = json.dumps(payload, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()[:16]


CACHE_KEY_EXCLUDE = {
    "batch_size",
    "force_recompute",
    "numba_threads_per_worker",
    "parallel_experiments_per_worker",
    "parallel_start_method",
    "parallel_workers",
    "particle_count",
    "use_cache",
}


def canonical_cache_setup(payload: dict) -> dict:
    semantic_setup = {key: value for key, value in payload.items() if key not in CACHE_KEY_EXCLUDE}
    return json.loads(json.dumps(semantic_setup, sort_keys=True, default=str))


CACHE_SETUP = canonical_cache_setup(setup)
CACHE_KEY = stable_hash(CACHE_SETUP | {"seed_base": SEED_BASE})


def sampler_strategy(strategy: str) -> str:
    if strategy == "adaptive":
        return ADAPTIVE_SAMPLER_STRATEGY
    return strategy


def sample_log_bayes(
    config: RabiExperimentConfig,
    strategy: str,
    hypothesis: str,
    n_experiments: int,
    seed: int,
    *,
    batch_size: int | None = None,
    true_drive_phases: np.ndarray | None = None,
) -> np.ndarray:
    return sample_rabi_log_bayes_factors(
        config,
        strategy=strategy,
        hypothesis=hypothesis,
        n_experiments=int(n_experiments),
        batch_size=BATCH_SIZE if batch_size is None else int(batch_size),
        seed=int(seed),
        method=SAMPLER_METHOD,
        true_drive_phases=true_drive_phases,
    )


def sample_h0(strategy: str, seed: int) -> np.ndarray:
    return sample_log_bayes(
        base_config,
        strategy=sampler_strategy(strategy),
        hypothesis="H0",
        n_experiments=N_NULL,
        seed=seed,
    )


def random_true_drive_phases(count: int, seed_base: int) -> np.ndarray:
    rng = np.random.default_rng(seed_base + 7919)
    return rng.uniform(0.0, 2.0 * np.pi, int(count))


def sample_h1_phase_averaged(strategy: str, seed_base: int) -> np.ndarray:
    if strategy == "fixed_z":
        return sample_log_bayes(
            base_config,
            strategy=sampler_strategy(strategy),
            hypothesis="H1",
            n_experiments=N_SIGNAL,
            seed=seed_base,
        )

    if PHASE_AVERAGE_MODE == "random_per_experiment":
        return sample_log_bayes(
            base_config,
            strategy=sampler_strategy(strategy),
            hypothesis="H1",
            n_experiments=N_SIGNAL,
            seed=seed_base,
            true_drive_phases=random_true_drive_phases(N_SIGNAL, seed_base),
        )

    per_phase = int(np.ceil(N_SIGNAL / len(TRUE_DRIVE_PHASES)))
    pieces = []
    for phase_index, drive_phase in enumerate(TRUE_DRIVE_PHASES):
        phase_config = replace(base_config, true_drive_phase=float(drive_phase))
        pieces.append(
            sample_log_bayes(
                phase_config,
                strategy=sampler_strategy(strategy),
                hypothesis="H1",
                n_experiments=per_phase,
                seed=seed_base + phase_index,
                batch_size=min(BATCH_SIZE, per_phase),
            )
        )
    return np.concatenate(pieces)[:N_SIGNAL]


def sample_cache_path(strategy: str, hypothesis: str) -> Path:
    filename = f"{OUTPUT_BASENAME}_{RUN_PRESET}_{CACHE_KEY}_{strategy}_{hypothesis}.npz"
    return CACHE_DIR / filename


def find_compatible_cache_path(strategy: str, hypothesis: str) -> Path | None:
    exact_path = sample_cache_path(strategy, hypothesis)
    if exact_path.exists():
        return exact_path

    pattern = f"{OUTPUT_BASENAME}_{RUN_PRESET}_*_{strategy}_{hypothesis}.npz"
    for candidate in sorted(CACHE_DIR.glob(pattern)):
        try:
            with np.load(candidate, allow_pickle=False) as data:
                if "setup" not in data.files:
                    continue
                cached_setup = json.loads(str(data["setup"].item()))
        except (OSError, ValueError, TypeError, json.JSONDecodeError):
            continue
        if canonical_cache_setup(cached_setup) == CACHE_SETUP:
            return candidate
    return None


_NUMBA_WARMED_UP = False
LOADED_CACHE_SETUPS: list[dict] = []
PREFIX_TRUE_DRIVE_PHASES: dict[str, np.ndarray] = {}
PREFIX_H0_KL: dict[str, np.ndarray] = {}
H1_DIAGNOSTICS: dict[str, dict[str, np.ndarray]] = {}
SAMPLE_SOURCES: dict[str, dict[str, str]] = {}
PREFIX_STRATEGY_NAMES = {
    "fixed_z": "fixed_z",
    "fixed_x": "fixed_x",
    "two_axis": "two_axis",
    "adaptive": "adaptive_l1",
}


def load_from_prefix_raw_cache(strategy: str, hypothesis: str) -> np.ndarray | None:
    if (
        not USE_CACHE
        or not USE_PREFIX_RAW_CACHE
        or FORCE_RECOMPUTE
        or RUN_PRESET != "final"
        or PHASE_AVERAGE_MODE != prefix_raw_setup.phase_average_mode
        or not PREFIX_RAW_CACHE_PATH.exists()
    ):
        return None

    prefix_strategy = PREFIX_STRATEGY_NAMES[strategy]
    template = prefix_raw_setup.base_template
    # Histogram overrides must also match the raw cache's model and policy.
    if BASE_PARTICLES.particle_mode != "grid" or any(
        getattr(base_config, key) != getattr(template, key)
        for key in ("sensing_time", "dead_time", "true_detuning", "initial_h1_probability")
    ) or any(
        getattr(base_config.readout, key) != getattr(template.readout, key)
        for key in ("contrast", "readout_error", "t1", "t2")
    ):
        return None
    grid = build_rabi_particle_grid(BASE_PARTICLES)
    if any(
        not np.array_equal(getattr(grid, key), getattr(prefix_raw_setup.particle_grid, key))
        for key in ("phase_amplitudes", "detunings", "drive_phases", "prior_weights")
    ):
        return None
    if sampler_strategy(strategy) != prefix_raw_setup.strategies[prefix_strategy].get("sampler_strategy", prefix_strategy):
        return None
    if strategy == "adaptive" and any(
        not np.array_equal(getattr(base_config, key), getattr(template, key))
        for key in ("candidate_angles", "candidate_polars")
    ):
        return None

    cumulative_kl = None
    with np.load(PREFIX_RAW_CACHE_PATH, allow_pickle=False) as data:
        cached_raw_setup = json.loads(str(data["raw_setup"].item()))
        # Other injected amplitudes do not affect this slice; check HIST_PHASE below.
        expected_raw_setup = prefix_raw_setup.raw_setup | {"phase_grid": cached_raw_setup["phase_grid"]}
        assert_matching_raw_setup(expected_raw_setup, cached_raw_setup)
        if not np.array_equal(data["phase_grid"], cached_raw_setup["phase_grid"]):
            raise ValueError("Prefix cache phase grid disagrees with its metadata")
        checkpoint_matches = np.flatnonzero(np.asarray(data["checkpoints"], dtype=int) == N_SHOTS)
        phase_matches = np.flatnonzero(np.isclose(np.asarray(data["phase_grid"], dtype=float), HIST_PHASE))
        if checkpoint_matches.size != 1 or (hypothesis == "H1" and phase_matches.size != 1):
            return None
        checkpoint_index = int(checkpoint_matches[0])
        if hypothesis == "H0":
            available = np.asarray(data[f"logB_h0__{prefix_strategy}"][:, checkpoint_index], dtype=float)
            requested = N_NULL
            kl_key = f"predictable_kl_h0__{prefix_strategy}"
            if kl_key in data.files:
                # Already cumulative at this checkpoint; use the same records as logB.
                cumulative_kl = np.asarray(data[kl_key][:, checkpoint_index], dtype=float)
                if cumulative_kl.shape != available.shape:
                    raise ValueError(f"Unpaired H0 logB and KL arrays for {strategy}")
        elif hypothesis == "H1":
            phase_index = int(phase_matches[0])
            available = np.asarray(data[f"logB_h1__{prefix_strategy}"][phase_index, :, checkpoint_index], dtype=float)
            PREFIX_TRUE_DRIVE_PHASES[strategy] = np.asarray(
                data[f"true_drive_phases_h1__{prefix_strategy}"][phase_index], dtype=float
            )
            requested = N_SIGNAL
        else:
            raise ValueError("hypothesis must be H0 or H1")

    if available.size < requested:
        return None
    if cumulative_kl is not None:
        PREFIX_H0_KL[strategy] = cumulative_kl[:requested].copy()
    setup["sample_source"] = "rabi_prefix_raw_data"
    return available[:requested].copy()


def warmup_numba_once() -> None:
    global _NUMBA_WARMED_UP
    if _NUMBA_WARMED_UP or not WARMUP_NUMBA or SAMPLER_METHOD != "numba":
        return
    warmup_config = replace(base_config, n_shots=min(16, N_SHOTS))
    try:
        sample_rabi_log_bayes_factors(
            warmup_config,
            strategy=ADAPTIVE_SAMPLER_STRATEGY,
            hypothesis="H0",
            n_experiments=2,
            batch_size=2,
            seed=SEED_BASE - 1,
            method=SAMPLER_METHOD,
        )
    except AssertionError as exc:
        raise RuntimeError(
            "Numba could not compile the Rabi sampler in this Python environment. "
            "Use a compatible cached result or a Python/Numba environment that supports recomputation."
        ) from exc
    _NUMBA_WARMED_UP = True


def load_or_generate(strategy: str, hypothesis: str, seed: int) -> np.ndarray:
    if hypothesis == "H1" and H1_KL_DIAGNOSTICS:
        return load_or_generate_h1_diagnostics(strategy, seed)
    path = sample_cache_path(strategy, hypothesis)
    cached_path = find_compatible_cache_path(strategy, hypothesis) if USE_CACHE else None
    if cached_path is not None and not FORCE_RECOMPUTE:
        with np.load(cached_path, allow_pickle=False) as data:
            samples = np.asarray(data["samples"], dtype=float)
            if "setup" in data.files:
                LOADED_CACHE_SETUPS.append(json.loads(str(data["setup"].item())))
        SAMPLE_SOURCES.setdefault(strategy, {})[hypothesis] = str(cached_path)
        print(f"loaded {hypothesis:>2s} {strategy:<9s}: {samples.size:7d} samples from {cached_path.name}")
        return samples

    prefix_samples = load_from_prefix_raw_cache(strategy, hypothesis)
    if prefix_samples is not None:
        SAMPLE_SOURCES.setdefault(strategy, {})[hypothesis] = str(PREFIX_RAW_CACHE_PATH)
        print(
            f"loaded {hypothesis:>2s} {strategy:<9s}: {prefix_samples.size:7d} samples "
            f"from {PREFIX_RAW_CACHE_PATH.name} at n={N_SHOTS}"
        )
        return prefix_samples

    if os.environ.get("RABI_ALLOW_SIMULATION", "0") != "1":
        raise FileNotFoundError("Missing histogram samples; set RABI_ALLOW_SIMULATION=1 to simulate.")
    warmup_numba_once()
    start = time.perf_counter()
    if hypothesis == "H0":
        samples = sample_h0(strategy, seed=seed)
    elif hypothesis == "H1":
        samples = sample_h1_phase_averaged(strategy, seed_base=seed)
    else:
        raise ValueError("hypothesis must be H0 or H1")
    elapsed = time.perf_counter() - start
    if USE_CACHE:
        np.savez_compressed(path, samples=samples, setup=json.dumps(setup, sort_keys=True, default=str))
    SAMPLE_SOURCES.setdefault(strategy, {})[hypothesis] = str(path) if USE_CACHE else "generated without cache"
    print(f"generated {hypothesis:>2s} {strategy:<9s}: {samples.size:7d} samples in {elapsed:7.1f} s")
    return samples


def mean_evidence_check(log_bayes: np.ndarray, prediction: np.ndarray, prefix: str) -> dict[str, float]:
    """Compare paired logBF and predictable drift samples, including their covariance."""
    log_bayes = np.asarray(log_bayes, dtype=float)
    prediction = np.asarray(prediction, dtype=float)
    if log_bayes.ndim != 1 or log_bayes.shape != prediction.shape or log_bayes.size < 2:
        raise ValueError("Need at least two paired logB and drift samples")
    if not np.all(np.isfinite(log_bayes)) or not np.all(np.isfinite(prediction)):
        raise ValueError("LogB and drift samples must be finite")
    residual = log_bayes - prediction
    sqrt_n = np.sqrt(log_bayes.size)
    residual_se = float(np.std(residual, ddof=1) / sqrt_n)
    residual_mean = float(np.mean(residual))
    return {
        f"{prefix}_mean_se": float(np.std(log_bayes, ddof=1) / sqrt_n),
        f"{prefix}_mean_kl_prediction": float(np.mean(prediction)),
        f"{prefix}_mean_kl_prediction_se": float(np.std(prediction, ddof=1) / sqrt_n),
        f"{prefix}_mean_residual": residual_mean,
        f"{prefix}_mean_residual_se": residual_se,
        f"{prefix}_mean_residual_z": residual_mean / residual_se if residual_se > 0.0 else np.nan,
    }


def null_mean_kl_check(log_bayes: np.ndarray, cumulative_kl: np.ndarray) -> dict[str, float]:
    """Eq. (A46): E0[log B] = -E0[sum KL]."""
    return mean_evidence_check(log_bayes, -np.asarray(cumulative_kl, dtype=float), "h0")


def signal_mean_kl_check(log_bayes: np.ndarray, null_kl: np.ndarray, predictive_kl: np.ndarray) -> dict[str, float]:
    """Eq. (A49): Etheta[log B] = Etheta[sum KL(true||null) - sum KL(true||mixture)]."""
    null_kl = np.asarray(null_kl, dtype=float)
    predictive_kl = np.asarray(predictive_kl, dtype=float)
    if np.shape(log_bayes) != null_kl.shape or null_kl.shape != predictive_kl.shape:
        raise ValueError("H1 logB and both KL arrays must refer to the same records")
    result = mean_evidence_check(log_bayes, null_kl - predictive_kl, "h1")
    result["h1_null_kl_mean"] = float(np.mean(null_kl))
    result["h1_predictive_kl_mean"] = float(np.mean(predictive_kl))
    return result


def load_or_generate_h1_diagnostics(strategy: str, seed: int) -> np.ndarray:
    if PARTICLE_MODE != "grid" or PHASE_AVERAGE_MODE != "random_per_experiment":
        raise ValueError("H1 KL diagnostics require grid particles and random_per_experiment phases; "
                         "set RABI_HIST_H1_KL=0 for the previous histogram-only path")
    phases = random_true_drive_phases(N_SIGNAL, seed)
    diagnostic_setup = {
        "version": "histogram-h1-kl-v1",
        "config": asdict(replace(base_config, seed=seed)),
        "sampler_strategy": sampler_strategy(strategy),
        "n_experiments": N_SIGNAL,
        "batch_size": BATCH_SIZE,
        "phase_average_mode": PHASE_AVERAGE_MODE,
        "true_drive_phases": phases.tolist(),
    }
    diagnostic_setup = json.loads(json.dumps(diagnostic_setup, sort_keys=True))
    path = CACHE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_h1_kl_{stable_hash(diagnostic_setup)}_{strategy}.npz"
    keys = ("log_bayes_factors", "signal_null_kl", "signal_predictive_kl")
    if USE_CACHE and path.exists() and not FORCE_RECOMPUTE:
        with np.load(path, allow_pickle=False) as data:
            if json.loads(str(data["setup"].item())) != diagnostic_setup:
                raise ValueError(f"H1 diagnostic cache setup mismatch: {path}")
            records = {key: np.asarray(data[key], dtype=float).copy() for key in keys}
            if not np.array_equal(data["true_drive_phases"], phases):
                raise ValueError(f"H1 diagnostic cache phase mismatch: {path}")
        print(f"loaded H1 {strategy:<9s}: {N_SIGNAL} paired logB/KL records from {path.name}")
    else:
        print(f"generating H1 {strategy}: {N_SIGNAL} paired logB/KL records at n={N_SHOTS}", flush=True)
        if strategy == "adaptive" and RUN_PRESET == "final":
            print("Full final settings (1000 records, 1024 shots): allow about 40-50 min on 8 threads", flush=True)
        if os.environ.get("RABI_ALLOW_SIMULATION", "0") != "1":
            raise FileNotFoundError("Missing histogram samples; set RABI_ALLOW_SIMULATION=1 to simulate.")
        warmup_numba_once()
        start = time.perf_counter()
        sampled = sample_rabi_diagnostics_at_checkpoints(
            base_config, strategy=sampler_strategy(strategy), hypothesis="H1",
            n_experiments=N_SIGNAL, checkpoints=[N_SHOTS], batch_size=BATCH_SIZE,
            seed=seed, true_drive_phases=phases,
        )
        records = {key: np.asarray(sampled[key][:, 0], dtype=float).copy() for key in keys}
        print(f"generated H1 {strategy}: {N_SIGNAL} paired records in {time.perf_counter() - start:.1f} s", flush=True)
    if any(values.shape != (N_SIGNAL,) or not np.all(np.isfinite(values)) for values in records.values()):
        raise ValueError(f"Invalid paired H1 diagnostic arrays for {strategy}")
    if USE_CACHE and (not path.exists() or FORCE_RECOMPUTE):
        np.savez_compressed(path, **records, true_drive_phases=phases,
                            setup=json.dumps(diagnostic_setup, sort_keys=True))
        print(f"saved H1 diagnostic cache {path.name}", flush=True)
    H1_DIAGNOSTICS[strategy] = records
    PREFIX_TRUE_DRIVE_PHASES[strategy] = phases
    SAMPLE_SOURCES.setdefault(strategy, {})["H1"] = str(path) if USE_CACHE else "generated without cache"
    return records["log_bayes_factors"]


In [ ]:
run_start = time.perf_counter()
null_samples: dict[str, np.ndarray] = {}
signal_samples: dict[str, np.ndarray] = {}
summary_rows = []
PREFIX_H0_KL.clear()
PREFIX_TRUE_DRIVE_PHASES.clear()
H1_DIAGNOSTICS.clear()
SAMPLE_SOURCES.clear()
LOADED_CACHE_SETUPS.clear()

for strategy_index, strategy in enumerate(STRATEGIES):
    h0 = load_or_generate(strategy, "H0", seed=SEED_BASE + 10_000 * strategy_index)
    h1 = load_or_generate(strategy, "H1", seed=SEED_BASE + 10_000 * strategy_index + 5_000)
    null_samples[strategy] = h0
    signal_samples[strategy] = h1

    threshold = null_threshold(h0, ALPHA)
    power = float(np.mean(h1 >= threshold))
    power_se = float(np.sqrt(max(power * (1.0 - power), 0.0) / len(h1)))
    empirical_alpha = float(np.mean(h0 >= threshold))
    summary_rows.append(
        {
            "strategy": strategy,
            "label": STRATEGIES[strategy]["label"],
            "threshold": threshold,
            "empirical_alpha": empirical_alpha,
            "power": power,
            "power_se": power_se,
            "beta": 1.0 - power,
            "n_h0": len(h0),
            "n_h1": len(h1),
            "h0_mean": float(np.mean(h0)),
            "h1_mean": float(np.mean(h1)),
            "h0_std": float(np.std(h0, ddof=1)),
            "h1_std": float(np.std(h1, ddof=1)),
        }
    )

    if strategy in PREFIX_H0_KL:
        summary_rows[-1].update(null_mean_kl_check(h0, PREFIX_H0_KL[strategy]))
    else:
        print(f"{strategy}: paired KL diagnostic unavailable; omitting the A46 mean check")

    if strategy in H1_DIAGNOSTICS:
        data = H1_DIAGNOSTICS[strategy]
        summary_rows[-1].update(signal_mean_kl_check(h1, data["signal_null_kl"], data["signal_predictive_kl"]))

setup["sample_sources"] = SAMPLE_SOURCES
h0_paths = {sources["H0"] for sources in SAMPLE_SOURCES.values()}
setup["h0_sample_source"] = "rabi_prefix_raw_data" if h0_paths == {str(PREFIX_RAW_CACHE_PATH)} else "histogram_samples"
if H1_KL_DIAGNOSTICS:
    setup["sample_source"] = "mixed"
    setup["h1_sample_source"] = "histogram_h1_diagnostics"
setup["h1_kl_diagnostics"] = H1_KL_DIAGNOSTICS

elapsed = time.perf_counter() - run_start
summary = pd.DataFrame(summary_rows)
display(summary)
print(f"total sampling/loading time: {elapsed:.1f} s")


In [ ]:
kl_columns = {
    "label": "Policy",
    "h0_mean": "Observed H0 mean",
    "h0_mean_se": "Observed SE",
    "h0_mean_kl_prediction": "KL mean prediction",
    "h0_mean_kl_prediction_se": "Prediction SE",
    "h0_mean_residual": "Paired residual",
    "h0_mean_residual_se": "Residual SE",
    "h0_mean_residual_z": "Residual / SE",
}
if "h0_mean_kl_prediction" in summary:
    display(summary[list(kl_columns)].rename(columns=kl_columns).round(4))

if "h1_mean_kl_prediction" in summary:
    h1_columns = {key.replace("h0_", "h1_"): value.replace("H0", "H1") for key, value in kl_columns.items()}
    h1_columns.update({"h1_null_kl_mean": "Mean KL(true || null)",
                       "h1_predictive_kl_mean": "Mean KL(true || mixture)"})
    display(summary[list(h1_columns)].rename(columns=h1_columns).round(4))

diagnostic_rows = []
for strategy in STRATEGIES:
    for hypothesis, samples in (("H0", null_samples[strategy]), ("H1", signal_samples[strategy])):
        diagnostic_rows.append(
            {
                "strategy": strategy,
                "hypothesis": hypothesis,
                "unique_round_6": int(np.unique(np.round(samples, 6)).size),
                "unique_round_3": int(np.unique(np.round(samples, 3)).size),
                "q01": float(np.quantile(samples, 0.01)),
                "q50": float(np.quantile(samples, 0.50)),
                "q99": float(np.quantile(samples, 0.99)),
            }
        )

diagnostics = pd.DataFrame(diagnostic_rows)
display(diagnostics)

phase_split_rows = []
for strategy_index, strategy in enumerate(STRATEGIES):
    if strategy == "fixed_z":
        continue
    threshold = float(summary.loc[summary["strategy"] == strategy, "threshold"].iloc[0])
    h1 = signal_samples[strategy]
    if PHASE_AVERAGE_MODE == "random_per_experiment":
        phases = PREFIX_TRUE_DRIVE_PHASES.get(strategy)
        if phases is None:
            seed_base = SEED_BASE + 10_000 * strategy_index + 5_000
            phases = random_true_drive_phases(len(h1), seed_base)
        phases = np.asarray(phases[: len(h1)], dtype=float)
        edges = np.linspace(0.0, 2.0 * np.pi, 13)
        for phase_index in range(len(edges) - 1):
            mask = (phases >= edges[phase_index]) & (phases < edges[phase_index + 1])
            chunk = h1[mask]
            if len(chunk) == 0:
                continue
            phase_value = 0.5 * (edges[phase_index] + edges[phase_index + 1])
            phase_split_rows.append(
                {
                    "strategy": strategy,
                    "phase_index": phase_index,
                    "drive_phase": float(phase_value),
                    "n": len(chunk),
                    "mean": float(np.mean(chunk)),
                    "power": float(np.mean(chunk >= threshold)),
                    "q90": float(np.quantile(chunk, 0.90)),
                    "q99": float(np.quantile(chunk, 0.99)),
                }
            )
    else:
        per_phase = int(np.ceil(N_SIGNAL / len(TRUE_DRIVE_PHASES)))
        for phase_index, drive_phase in enumerate(TRUE_DRIVE_PHASES):
            chunk = h1[phase_index * per_phase : min((phase_index + 1) * per_phase, len(h1))]
            if len(chunk) == 0:
                continue
            phase_split_rows.append(
                {
                    "strategy": strategy,
                    "phase_index": phase_index,
                    "drive_phase": float(np.mod(drive_phase, 2.0 * np.pi)),
                    "n": len(chunk),
                    "mean": float(np.mean(chunk)),
                    "power": float(np.mean(chunk >= threshold)),
                    "q90": float(np.quantile(chunk, 0.90)),
                    "q99": float(np.quantile(chunk, 0.99)),
                }
            )

phase_split = pd.DataFrame(phase_split_rows)
display(phase_split.head(min(len(phase_split), 24)))


In [ ]:
mpl.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 350,
        "font.size": 13,
        "axes.labelsize": 13,
        "axes.titlesize": 13,
        "legend.fontsize": 11,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.linewidth": 0.9,
        "mathtext.fontset": "stix",
        "font.family": "STIXGeneral",
    }
)

PAPER_STRATEGIES = ("fixed_z", "adaptive")
PAPER_LABELS = {"fixed_z": r"Fixed $z$", "adaptive": "Adaptive"}
PAPER_POWER_SYMBOLS = {"fixed_z": r"\overline{P}^{z}", "adaptive": r"\overline{P}^{\mathrm{adapt}}"}
all_values = np.concatenate(
    [
        *(null_samples[strategy] for strategy in PAPER_STRATEGIES),
        *(signal_samples[strategy] for strategy in PAPER_STRATEGIES),
    ]
)
all_thresholds = summary.loc[summary["strategy"].isin(PAPER_STRATEGIES), "threshold"].to_numpy(dtype=float)
lo, hi = np.percentile(all_values, [0.15, 99.85])
lo = min(float(lo), float(np.min(all_thresholds)) - 0.5)
hi = max(float(hi), float(np.max(all_thresholds)) + 0.5)
bins = np.linspace(lo, hi, BIN_COUNT + 1)
paper_sample_count = max(
    *(len(null_samples[strategy]) for strategy in PAPER_STRATEGIES),
    *(len(signal_samples[strategy]) for strategy in PAPER_STRATEGIES),
)
paper_y_max = 10.0 ** math.ceil(math.log10(paper_sample_count))
paper_x_max = max(7.5, math.ceil((float(np.max(all_values)) + 0.25) * 2.0) / 2.0)

fig, axes = plt.subplots(len(PAPER_STRATEGIES), 1, figsize=(5.2, 5.3), sharex=True)
axes = np.atleast_1d(axes)

for panel_index, (ax, strategy) in enumerate(zip(axes, PAPER_STRATEGIES), start=1):
    meta = STRATEGIES[strategy]
    row = summary.loc[summary["strategy"] == strategy].iloc[0]
    h0 = null_samples[strategy]
    h1 = signal_samples[strategy]

    ax.hist(h0, bins=bins, histtype="stepfilled", alpha=0.22, color="#2F5D7C", label=r"$H_0$")
    ax.hist(h0, bins=bins, histtype="step", linewidth=1.45, color="#2F5D7C")
    ax.hist(h1, bins=bins, histtype="stepfilled", alpha=0.20, color="#C2702C", label="signal")
    ax.hist(h1, bins=bins, histtype="step", linewidth=1.45, color="#C2702C")
    ax.axvline(row["threshold"], color="black", linestyle="--", linewidth=1.15, label=rf"$\alpha={ALPHA:.2f}$")

    if pd.notna(row.get("h0_mean_kl_prediction", np.nan)):
        kl_mean = row["h0_mean_kl_prediction"]
        ax.axvline(kl_mean, color="#2F5D7C", linestyle=":", linewidth=2.3, zorder=4)
        ax.axvline(row["h0_mean"], color="#2F5D7C", linestyle="-.", linewidth=1.2,
                   label=r"$H_0$ mean", zorder=4)

    if pd.notna(row.get("h1_mean_kl_prediction", np.nan)):
        kl_mean = row["h1_mean_kl_prediction"]
        ax.axvline(kl_mean, color="#C2702C", linestyle=":", linewidth=2.3, zorder=4)
        ax.axvline(row["h1_mean"], color="#C2702C", linestyle="-.", linewidth=1.2,
                   label=r"$H_1$ mean", zorder=4)

    ax.set_title(
        rf"({chr(96 + panel_index)}) {PAPER_LABELS[strategy]}"
        + rf"   ${PAPER_POWER_SYMBOLS[strategy]}={row['power']:.3f}\pm{row['power_se']:.3f}$",
        loc="left",
    )
    legend = ax.legend(
        loc="upper right",
        frameon=False,
        fontsize=10,
        labelspacing=0.3,
        handlelength=1.9,
        borderaxespad=0.8,
    )
    ax.set_yscale("log")
    ax.set_ylim(1.0, paper_y_max)
    ax.set_box_aspect(0.48)
    ax.grid(alpha=0.18, linewidth=0.6)
    ax.set_ylabel("pseudo experiments")

axes[-1].set_xlim(-2.5, paper_x_max)
axes[-1].set_xlabel(r"$\log B_{10,n}$")
fig.tight_layout(pad=0.5, h_pad=1.2)

if SAVE_FIGURES:
    pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}.pdf"
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"saved {pdf_path}")

plt.show()

In [ ]:
CONVENTIONAL_STRATEGIES = {
    "fixed_z": {"label": r"Fixed $z$ count", "color": STRATEGIES["fixed_z"]["color"]},
    "fixed_x": {"label": r"Fixed $x$ count", "color": STRATEGIES["fixed_x"]["color"]},
    "two_axis": {"label": r"Fixed $x/y$ quadrature", "color": STRATEGIES["two_axis"]["color"]},
}
CONVENTIONAL_PRESETS = {
    name: {
        "n_shots": values["n_shots"],
        "n_null": values["n_null"],
        "n_signal": values["n_signal"],
        "bin_count": values["bin_count"],
        "phase_average_count": values["phase_average_count"],
        "theory_phase_count": max(1024, values["phase_average_count"]),
    }
    for name, values in PRESETS.items()
}
CONVENTIONAL_PRESETS["fixed_axis_smooth"] = {
    "n_shots": N_SHOTS,
    "n_null": 200_000,
    "n_signal": 200_000,
    "bin_count": 120,
    "phase_average_count": 4096,
    "theory_phase_count": 8192,
}

# This section can be rerun independently of the log-Bayes Monte Carlo above.
# In Jupyter, run a small cell such as
#     RABI_HIST_CONVENTIONAL_PRESET = "fixed_axis_smooth"
# before this cell, or edit the environment variable with the same name.
RABI_HIST_CONVENTIONAL_PRESET = os.environ.get(
    "RABI_HIST_CONVENTIONAL_PRESET",
    globals().get("RABI_HIST_CONVENTIONAL_PRESET", "fixed_axis_smooth"),
)
def conventional_setting(name: str, default):
    return globals().get(name, os.environ.get(name, default))


CONVENTIONAL_PRESET = str(conventional_setting("RABI_HIST_CONVENTIONAL_PRESET", RUN_PRESET))
if CONVENTIONAL_PRESET not in CONVENTIONAL_PRESETS:
    raise ValueError(f"Unknown CONVENTIONAL_PRESET={CONVENTIONAL_PRESET!r}; choose one of {sorted(CONVENTIONAL_PRESETS)}")
CONVENTIONAL_N_SHOTS = int(conventional_setting("RABI_HIST_CONVENTIONAL_N_SHOTS", CONVENTIONAL_PRESETS[CONVENTIONAL_PRESET]["n_shots"]))
CONVENTIONAL_N_NULL = int(conventional_setting("RABI_HIST_CONVENTIONAL_N_NULL", CONVENTIONAL_PRESETS[CONVENTIONAL_PRESET]["n_null"]))
CONVENTIONAL_N_SIGNAL = int(conventional_setting("RABI_HIST_CONVENTIONAL_N_SIGNAL", CONVENTIONAL_PRESETS[CONVENTIONAL_PRESET]["n_signal"]))
CONVENTIONAL_BIN_COUNT = int(conventional_setting("RABI_HIST_CONVENTIONAL_BIN_COUNT", CONVENTIONAL_PRESETS[CONVENTIONAL_PRESET]["bin_count"]))
CONVENTIONAL_HIST_PHASE = float(conventional_setting("RABI_HIST_CONVENTIONAL_PHI_R", HIST_PHASE))
CONVENTIONAL_PHASE_AVERAGE_MODE = str(conventional_setting("RABI_HIST_CONVENTIONAL_PHASE_MODE", PHASE_AVERAGE_MODE))
CONVENTIONAL_PHASE_AVERAGE_COUNT = int(conventional_setting("RABI_HIST_CONVENTIONAL_PHASE_COUNT", CONVENTIONAL_PRESETS[CONVENTIONAL_PRESET]["phase_average_count"]))
CONVENTIONAL_THEORY_PHASE_COUNT = int(conventional_setting("RABI_HIST_CONVENTIONAL_THEORY_PHASE_COUNT", CONVENTIONAL_PRESETS[CONVENTIONAL_PRESET]["theory_phase_count"]))

if CONVENTIONAL_PHASE_AVERAGE_MODE == "uniform_grid":
    CONVENTIONAL_TRUE_DRIVE_PHASES = tuple(np.linspace(0.0, 2.0 * np.pi, CONVENTIONAL_PHASE_AVERAGE_COUNT, endpoint=False))
elif CONVENTIONAL_PHASE_AVERAGE_MODE == "uniform_midpoint":
    conventional_phase_step = 2.0 * np.pi / CONVENTIONAL_PHASE_AVERAGE_COUNT
    CONVENTIONAL_TRUE_DRIVE_PHASES = tuple(np.linspace(0.0, 2.0 * np.pi, CONVENTIONAL_PHASE_AVERAGE_COUNT, endpoint=False) + 0.5 * conventional_phase_step)
elif CONVENTIONAL_PHASE_AVERAGE_MODE == "random_grid":
    conventional_phase_rng = np.random.default_rng(SEED_BASE + 99_731)
    CONVENTIONAL_TRUE_DRIVE_PHASES = tuple(np.sort(conventional_phase_rng.uniform(0.0, 2.0 * np.pi, CONVENTIONAL_PHASE_AVERAGE_COUNT)))
elif CONVENTIONAL_PHASE_AVERAGE_MODE == "random_per_experiment":
    CONVENTIONAL_TRUE_DRIVE_PHASES = tuple()
else:
    raise ValueError("CONVENTIONAL_PHASE_AVERAGE_MODE must be 'uniform_grid', 'uniform_midpoint', 'random_grid', or 'random_per_experiment'")

print(
    "conventional count setup:",
    {
        "preset": CONVENTIONAL_PRESET,
        "n_shots": CONVENTIONAL_N_SHOTS,
        "n_null": CONVENTIONAL_N_NULL,
        "n_signal": CONVENTIONAL_N_SIGNAL,
        "Phi_R": CONVENTIONAL_HIST_PHASE,
        "phase_average_mode": CONVENTIONAL_PHASE_AVERAGE_MODE,
        "phase_average_count": CONVENTIONAL_PHASE_AVERAGE_COUNT,
        "theory_phase_count": CONVENTIONAL_THEORY_PHASE_COUNT,
    },
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CONVENTIONAL_READOUT = RabiReadoutConfig(
    contrast=CONTRAST,
    readout_error=READOUT_ERROR,
    t1=T1_TIME,
    t2=T2_TIME,
)
from scipy.stats import binom, chi2, ncx2

if TRUE_DETUNING != 0.0:
    raise ValueError("Conventional iid count tests require TRUE_DETUNING=0")
if CONVENTIONAL_N_SHOTS < 2:
    raise ValueError("The x/y count test requires at least two shots")


def conventional_random_true_drive_phases(count: int, seed: int) -> np.ndarray:
    return np.random.default_rng(seed).uniform(0.0, 2.0 * np.pi, int(count))


def conventional_axis_probability(axis: str, readout_angle: float, phase_amplitude: float, drive_phases: np.ndarray) -> np.ndarray:
    drive_phases = np.asarray(drive_phases, dtype=float)
    probability = np.asarray(
        rabi_probability_one(
            phase_amplitude,
            TRUE_DETUNING,
            drive_phases,
            0.0,
            SENSING_TIME,
            axis,
            readout_angle,
            CONVENTIONAL_READOUT,
        ),
        dtype=float,
    )
    if probability.shape == ():
        probability = np.full(drive_phases.shape, float(probability))
    return np.clip(probability, 1.0e-12, 1.0 - 1.0e-12)


def conventional_count_samples_for_phases(strategy: str, phase_amplitude: float, drive_phases: np.ndarray, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    drive_phases = np.asarray(drive_phases, dtype=float)
    if strategy == "fixed_z":
        p_plus = conventional_axis_probability("z", 0.0, phase_amplitude, drive_phases)
        return rng.binomial(CONVENTIONAL_N_SHOTS, p_plus).astype(int)
    if strategy == "fixed_x":
        p_plus = conventional_axis_probability("transverse", 0.0, phase_amplitude, drive_phases)
        return rng.binomial(CONVENTIONAL_N_SHOTS, p_plus).astype(int)
    if strategy == "two_axis":
        n_x = (CONVENTIONAL_N_SHOTS + 1) // 2
        n_y = CONVENTIONAL_N_SHOTS // 2
        p_x = conventional_axis_probability("transverse", 0.0, phase_amplitude, drive_phases)
        p_y = conventional_axis_probability("transverse", 0.5 * math.pi, phase_amplitude, drive_phases)
        return np.column_stack([rng.binomial(n_x, p_x), rng.binomial(n_y, p_y)]).astype(int)
    raise ValueError(f"unknown conventional strategy: {strategy}")


def conventional_h0_count_samples(strategy: str, seed: int) -> np.ndarray:
    return conventional_count_samples_for_phases(strategy, 0.0, np.zeros(CONVENTIONAL_N_NULL), seed)


def conventional_h1_count_samples(strategy: str, seed: int) -> np.ndarray:
    if strategy == "fixed_z":
        return conventional_count_samples_for_phases(strategy, CONVENTIONAL_HIST_PHASE, np.zeros(CONVENTIONAL_N_SIGNAL), seed)
    if CONVENTIONAL_PHASE_AVERAGE_MODE == "random_per_experiment":
        phases = conventional_random_true_drive_phases(CONVENTIONAL_N_SIGNAL, seed)
        return conventional_count_samples_for_phases(strategy, CONVENTIONAL_HIST_PHASE, phases, seed + 1)

    pieces = []
    remaining = CONVENTIONAL_N_SIGNAL
    per_phase = int(math.ceil(CONVENTIONAL_N_SIGNAL / max(1, len(CONVENTIONAL_TRUE_DRIVE_PHASES))))
    for phase_index, drive_phase in enumerate(CONVENTIONAL_TRUE_DRIVE_PHASES):
        if remaining <= 0:
            break
        count = min(per_phase, remaining)
        phases = np.full(count, drive_phase)
        pieces.append(conventional_count_samples_for_phases(strategy, CONVENTIONAL_HIST_PHASE, phases, seed + 101 * phase_index))
        remaining -= count
    return np.concatenate(pieces) if pieces else np.empty(0, dtype=int)


def conventional_h0_center(strategy: str) -> float:
    phase = np.zeros(1)
    if strategy == "fixed_z":
        return float(CONVENTIONAL_N_SHOTS * conventional_axis_probability("z", 0.0, 0.0, phase)[0])
    if strategy == "fixed_x":
        return float(CONVENTIONAL_N_SHOTS * conventional_axis_probability("transverse", 0.0, 0.0, phase)[0])
    if strategy == "two_axis":
        n_x = (CONVENTIONAL_N_SHOTS + 1) // 2
        n_y = CONVENTIONAL_N_SHOTS // 2
        p_x = conventional_axis_probability("transverse", 0.0, 0.0, phase)[0]
        p_y = conventional_axis_probability("transverse", 0.5 * math.pi, 0.0, phase)[0]
        return float(n_x * p_x + n_y * p_y)
    raise ValueError(f"unknown conventional strategy: {strategy}")


def conventional_xy_h0_moments() -> tuple[np.ndarray, np.ndarray]:
    phase = np.zeros(1)
    n_x = (CONVENTIONAL_N_SHOTS + 1) // 2
    n_y = CONVENTIONAL_N_SHOTS // 2
    p_x0 = float(conventional_axis_probability("transverse", 0.0, 0.0, phase)[0])
    p_y0 = float(conventional_axis_probability("transverse", 0.5 * math.pi, 0.0, phase)[0])
    center = np.asarray([n_x * p_x0, n_y * p_y0], dtype=float)
    variance = np.asarray([n_x * p_x0 * (1.0 - p_x0), n_y * p_y0 * (1.0 - p_y0)], dtype=float)
    return center, np.maximum(variance, 1.0e-12)

def conventional_xy_quadrature_statistic(samples_xy: np.ndarray, center: np.ndarray, variance: np.ndarray) -> np.ndarray:
    delta = np.asarray(samples_xy, dtype=float) - center[None, :]
    return np.sum(delta * delta / variance[None, :], axis=1)


def randomized_threshold(values: np.ndarray, probabilities: np.ndarray, alpha: float) -> tuple[float, float]:
    """Upper-tail test of exact size alpha on a finite null support."""
    support, inverse = np.unique(np.asarray(values).ravel(), return_inverse=True)
    masses = np.bincount(inverse, weights=np.asarray(probabilities).ravel())
    masses /= masses.sum()
    index = int(np.searchsorted(np.cumsum(masses), 1.0 - alpha))
    above = float(masses[index + 1:].sum())
    gamma = float(np.clip((alpha - above) / masses[index], 0.0, 1.0))
    return float(support[index]), gamma


def rejection_probabilities(statistic: np.ndarray, boundary: float, gamma: float) -> np.ndarray:
    """Integrate out the boundary coin instead of adding extra Monte Carlo noise."""
    statistic = np.asarray(statistic, dtype=float)
    return (statistic > boundary).astype(float) + gamma * (statistic == boundary)


def conventional_statistic(strategy: str, counts: np.ndarray) -> np.ndarray:
    if strategy == "two_axis":
        center, variance = conventional_xy_h0_moments()
        return conventional_xy_quadrature_statistic(counts, center, variance)
    counts = np.asarray(counts, dtype=float)
    return counts if strategy == "fixed_z" else np.abs(counts - conventional_h0_center(strategy))


def conventional_null_support(strategy: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Counts, test statistics and their exact binomial null masses."""
    if strategy == "two_axis":
        n_x, n_y = (CONVENTIONAL_N_SHOTS + 1) // 2, CONVENTIONAL_N_SHOTS // 2
        k_x, k_y = np.meshgrid(np.arange(n_x + 1), np.arange(n_y + 1), indexing="ij")
        counts = np.column_stack([k_x.ravel(), k_y.ravel()])
        center, _ = conventional_xy_h0_moments()
        masses = (binom.pmf(k_x, n_x, center[0] / n_x)
                  * binom.pmf(k_y, n_y, center[1] / n_y)).ravel()
    else:
        counts = np.arange(CONVENTIONAL_N_SHOTS + 1)
        p0 = conventional_h0_center(strategy) / CONVENTIONAL_N_SHOTS
        masses = binom.pmf(counts, CONVENTIONAL_N_SHOTS, p0)
    return counts, conventional_statistic(strategy, counts), masses


def conventional_theory_phases() -> np.ndarray:
    return np.linspace(0.0, 2.0 * np.pi, CONVENTIONAL_THEORY_PHASE_COUNT, endpoint=False)


def conventional_exact_power(strategy: str, boundary: float, gamma: float) -> float:
    """Exact count probabilities; transverse power averaged by phase quadrature."""
    if strategy == "fixed_z":
        p1 = conventional_axis_probability("z", 0.0, CONVENTIONAL_HIST_PHASE, np.zeros(1))[0]
        return float(binom.sf(boundary, CONVENTIONAL_N_SHOTS, p1)
                     + gamma * binom.pmf(boundary, CONVENTIONAL_N_SHOTS, p1))
    phases = conventional_theory_phases()
    counts, statistic, _ = conventional_null_support(strategy)
    acceptance = 1.0 - rejection_probabilities(statistic, boundary, gamma)
    p_x = conventional_axis_probability("transverse", 0.0, CONVENTIONAL_HIST_PHASE, phases)
    if strategy == "fixed_x":
        keep = acceptance > 0.0
        beta = binom.pmf(counts[keep][None, :], CONVENTIONAL_N_SHOTS, p_x[:, None]) @ acceptance[keep]
    else:
        n_x, n_y = (CONVENTIONAL_N_SHOTS + 1) // 2, CONVENTIONAL_N_SHOTS // 2
        acceptance = acceptance.reshape(n_x + 1, n_y + 1)
        # Only the small central acceptance region contributes to beta.
        k_x = np.flatnonzero(np.any(acceptance > 0.0, axis=1))
        k_y = np.flatnonzero(np.any(acceptance > 0.0, axis=0))
        p_y = conventional_axis_probability("transverse", 0.5 * math.pi, CONVENTIONAL_HIST_PHASE, phases)
        px_mass = binom.pmf(k_x[None, :], n_x, p_x[:, None])
        py_mass = binom.pmf(k_y[None, :], n_y, p_y[:, None])
        beta = np.sum((px_mass @ acceptance[np.ix_(k_x, k_y)]) * py_mass, axis=1)
    return float(1.0 - np.mean(beta))


def conventional_detection_row(strategy: str, h0_counts: np.ndarray, h1_counts: np.ndarray) -> dict:
    _, null_statistic, null_mass = conventional_null_support(strategy)
    threshold, gamma = randomized_threshold(null_statistic, null_mass, ALPHA)
    reject_h0 = rejection_probabilities(conventional_statistic(strategy, h0_counts), threshold, gamma)
    reject_h1 = rejection_probabilities(conventional_statistic(strategy, h1_counts), threshold, gamma)
    center = conventional_h0_center(strategy) if strategy != "two_axis" else 2.0
    return {
        "strategy": strategy,
        "label": CONVENTIONAL_STRATEGIES[strategy]["label"],
        "test_type": {"fixed_z": "randomized upper-tail count",
                      "fixed_x": "randomized two-sided count deviation",
                      "two_axis": "randomized two-quadrature radial deviation"}[strategy],
        "h0_center": center,
        "threshold_statistic": threshold,
        "boundary_reject_probability": gamma,
        "lower_count_cut": center - threshold if strategy == "fixed_x" else np.nan,
        "upper_count_cut": (center + threshold if strategy == "fixed_x" else
                            threshold if strategy == "fixed_z" else np.nan),
        "empirical_alpha": float(np.mean(reject_h0)),
        "power": float(np.mean(reject_h1)),
        # The boundary coin is integrated out: use the SE of these bounded weights.
        "power_se": float(np.std(reject_h1, ddof=1) / np.sqrt(len(reject_h1))),
        "n_signal": len(h1_counts),
        "n_null": len(h0_counts),
        "theory_alpha": float(np.dot(null_mass, rejection_probabilities(null_statistic, threshold, gamma))),
        "theory_power": conventional_exact_power(strategy, threshold, gamma),
        "theory_method": "exact binomial" if strategy == "fixed_z" else "phase-averaged exact binomial",
    }


conventional_null_samples = {}
conventional_signal_samples = {}
conventional_rows = []
for strategy_index, strategy in enumerate(CONVENTIONAL_STRATEGIES):
    h0_seed = SEED_BASE + (500_001 if strategy == "two_axis" else 300_000 + 17 * strategy_index)
    h1_seed = SEED_BASE + (500_101 if strategy == "two_axis" else 301_000 + 17 * strategy_index)
    h0_counts = conventional_h0_count_samples(strategy, h0_seed)
    h1_counts = conventional_h1_count_samples(strategy, h1_seed)
    conventional_null_samples[strategy] = h0_counts
    conventional_signal_samples[strategy] = h1_counts
    conventional_rows.append(conventional_detection_row(strategy, h0_counts, h1_counts))

conventional_summary = pd.DataFrame(conventional_rows)
display(conventional_summary)

fig_conv, axes_conv = plt.subplots(1, 3, figsize=(8.1, 3.35), sharey=False)
for ax, strategy in zip(axes_conv, CONVENTIONAL_STRATEGIES):
    meta = CONVENTIONAL_STRATEGIES[strategy]
    row = conventional_summary.loc[conventional_summary["strategy"] == strategy].iloc[0]
    h0_counts = conventional_null_samples[strategy]
    h1_counts = conventional_signal_samples[strategy]
    if strategy == "two_axis":
        h0_counts = conventional_statistic(strategy, h0_counts)
        h1_counts = conventional_statistic(strategy, h1_counts)
    lo_count = float(min(np.min(h0_counts), np.min(h1_counts)))
    hi_count = float(max(np.max(h0_counts), np.max(h1_counts)))
    local_bin_count = CONVENTIONAL_BIN_COUNT if strategy == "two_axis" else min(CONVENTIONAL_BIN_COUNT, max(12, int(hi_count - lo_count + 1)))
    count_bins = np.linspace(lo_count - 0.5, hi_count + 0.5, local_bin_count + 1)

    ax.hist(h0_counts, bins=count_bins, histtype="stepfilled", alpha=0.22, color="#2F5D7C", label=r"$H_0$")
    ax.hist(h0_counts, bins=count_bins, histtype="step", linewidth=1.35, color="#2F5D7C")
    ax.hist(h1_counts, bins=count_bins, histtype="stepfilled", alpha=0.20, color="#C2702C", label=rf"$H_1$, $\Phi_R={CONVENTIONAL_HIST_PHASE:.2f}$")
    ax.hist(h1_counts, bins=count_bins, histtype="step", linewidth=1.35, color="#C2702C")
    ax.axvline(row["h0_center"], color="0.35", linestyle=":", linewidth=1.0, label=r"$H_0$ mean")
    if strategy in {"fixed_z", "two_axis"}:
        ax.axvline(row["threshold_statistic"], color="black", linestyle="--", linewidth=1.1, label="randomized boundary")
    else:
        ax.axvline(row["lower_count_cut"], color="black", linestyle="--", linewidth=1.1, label="randomized boundaries")
        ax.axvline(row["upper_count_cut"], color="black", linestyle="--", linewidth=1.1)
    ax.set_title(meta["label"])
    ax.set_xlabel(r"$Q_{xy}$" if strategy == "two_axis" else r"outcome-1 count $K_1$")
    ax.set_ylim(0.0, 1.7 * ax.get_ylim()[1])
    ax.text(
        0.03,
        0.95,
        rf"MC $1-\beta={row['power']:.3f}\pm{row['power_se']:.3f}$"
        + "\n"
        + rf"theory $1-\beta={row['theory_power']:.3f}$"
        + "\n"
        + rf"$\hat\alpha={row['empirical_alpha']:.3f}$, $\alpha_{{\rm th}}={row['theory_alpha']:.3f}$"
        + "\n" + rf"boundary rejection $\gamma={row['boundary_reject_probability']:.3f}$",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8.0,
        bbox={"boxstyle": "round,pad=0.25", "facecolor": "white", "edgecolor": "0.8", "alpha": 0.88},
    )
    ax.grid(alpha=0.18, linewidth=0.6)

axes_conv[0].set_ylabel("pseudo experiments / bin")
handles, labels = axes_conv[0].get_legend_handles_labels()
fig_conv.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.05))
fig_conv.suptitle(rf"Exact-size randomized count tests, $n_{{\rm shot}}={CONVENTIONAL_N_SHOTS}$, $\Phi_R={CONVENTIONAL_HIST_PHASE:.2f}$", y=1.12, fontsize=12)
fig_conv.tight_layout(rect=(0.0, 0.0, 1.0, 0.98))

if SAVE_FIGURES:
    conv_pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{CONVENTIONAL_PRESET}_conventional_counts.pdf"
    conv_summary_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{CONVENTIONAL_PRESET}_conventional_counts_summary.csv"
    fig_conv.savefig(conv_pdf_path, bbox_inches="tight")
    conventional_summary.to_csv(conv_summary_path, index=False)
    print(f"saved {conv_pdf_path}")
    print(f"saved {conv_summary_path}")

plt.show()


In [ ]:
# Reuse the joint counts and exact-size test from the preceding figure.
xy_h0 = conventional_null_samples["two_axis"]
xy_h1 = conventional_signal_samples["two_axis"]
xy_count_row = conventional_summary.set_index("strategy").loc["two_axis"]
xy_center, xy_variance = conventional_xy_h0_moments()
q_h0 = conventional_xy_quadrature_statistic(xy_h0, xy_center, xy_variance)
q_h1 = conventional_xy_quadrature_statistic(xy_h1, xy_center, xy_variance)
q_threshold = float(xy_count_row["threshold_statistic"])
q_boundary_reject_probability = float(xy_count_row["boundary_reject_probability"])
xy_theory_phases = conventional_theory_phases()
xy_n_x = (CONVENTIONAL_N_SHOTS + 1) // 2
xy_n_y = CONVENTIONAL_N_SHOTS // 2
xy_p_x1 = conventional_axis_probability("transverse", 0.0, CONVENTIONAL_HIST_PHASE, xy_theory_phases)
xy_p_y1 = conventional_axis_probability("transverse", 0.5 * math.pi, CONVENTIONAL_HIST_PHASE, xy_theory_phases)
xy_mean_shift_x = xy_n_x * xy_p_x1 - xy_center[0]
xy_mean_shift_y = xy_n_y * xy_p_y1 - xy_center[1]
xy_noncentrality = xy_mean_shift_x * xy_mean_shift_x / xy_variance[0] + xy_mean_shift_y * xy_mean_shift_y / xy_variance[1]
xy_lambda = float(np.mean(xy_noncentrality))
q_chi2_threshold = float(chi2.isf(ALPHA, df=2))
xy_theory_alpha_boundary_chi2 = float(chi2.sf(q_threshold, df=2))
xy_theory_power_boundary_chi2 = float(np.mean(ncx2.sf(q_threshold, df=2, nc=xy_noncentrality)))
xy_theory_power_chi2_cut = float(np.mean(ncx2.sf(q_chi2_threshold, df=2, nc=xy_noncentrality)))
xy_reject_h0 = rejection_probabilities(q_h0, q_threshold, q_boundary_reject_probability)
xy_reject_h1 = rejection_probabilities(q_h1, q_threshold, q_boundary_reject_probability)
xy_power = float(np.mean(xy_reject_h1))
xy_empirical_alpha = float(np.mean(xy_reject_h0))
xy_power_se = float(np.std(xy_reject_h1, ddof=1) / np.sqrt(len(q_h1)))

xy_quadrature_summary = pd.DataFrame(
    [
        {
            "strategy": "fixed_x_y_quadrature",
            "label": r"Fixed $x/y$ quadrature",
            "test_type": "two-quadrature radial count deviation",
            "kx_h0_center": float(xy_center[0]),
            "ky_h0_center": float(xy_center[1]),
            "kx_h0_variance": float(xy_variance[0]),
            "ky_h0_variance": float(xy_variance[1]),
            "q_threshold": float(q_threshold),
            "q_chi2_threshold": q_chi2_threshold,
            "boundary_reject_probability": q_boundary_reject_probability,
            "theory_alpha_exact": float(xy_count_row["theory_alpha"]),
            "theory_power_exact": float(xy_count_row["theory_power"]),
            "empirical_alpha": xy_empirical_alpha,
            "power": xy_power,
            "power_se": xy_power_se,
            "theory_alpha_boundary_chi2": xy_theory_alpha_boundary_chi2,
            "theory_power_boundary_chi2": xy_theory_power_boundary_chi2,
            "theory_power_chi2_cut": xy_theory_power_chi2_cut,
            "theory_method": "exact binomial (randomized), plus noncentral chi-square approximations",
            "n_signal": len(q_h1),
            "n_null": len(q_h0),
        }
    ]
)
display(xy_quadrature_summary)

combined_xy = np.vstack([xy_h0, xy_h1])
x_low, x_high = np.percentile(combined_xy[:, 0], [0.5, 99.5])
y_low, y_high = np.percentile(combined_xy[:, 1], [0.5, 99.5])
noncentrality_t = np.linspace(0.0, 2.0 * np.pi, 400)
noncentrality_radius = math.sqrt(xy_lambda)
noncentrality_x = xy_center[0] + math.sqrt(xy_variance[0]) * noncentrality_radius * np.cos(noncentrality_t)
noncentrality_y = xy_center[1] + math.sqrt(xy_variance[1]) * noncentrality_radius * np.sin(noncentrality_t)
x_low = min(float(x_low), float(np.min(noncentrality_x)) - 2.0)
x_high = max(float(x_high), float(np.max(noncentrality_x)) + 2.0)
y_low = min(float(y_low), float(np.min(noncentrality_y)) - 2.0)
y_high = max(float(y_high), float(np.max(noncentrality_y)) + 2.0)
xy_bins = min(70, max(24, CONVENTIONAL_BIN_COUNT))
x_edges = np.linspace(x_low - 0.5, x_high + 0.5, xy_bins + 1)
y_edges = np.linspace(y_low - 0.5, y_high + 0.5, xy_bins + 1)

fig_xy = plt.figure(figsize=(7.2, 7.2), constrained_layout=True)
outer_grid = fig_xy.add_gridspec(2, 1, height_ratios=[1.0, 0.72])
top_grid = outer_grid[0].subgridspec(1, 3, width_ratios=[1.0, 1.0, 0.055], wspace=0.06)
axes_xy = np.asarray(
    [
        fig_xy.add_subplot(top_grid[0, 0]),
        fig_xy.add_subplot(top_grid[0, 1]),
        fig_xy.add_subplot(outer_grid[1, 0]),
    ],
    dtype=object,
)
axes_xy[1].sharex(axes_xy[0])
axes_xy[1].sharey(axes_xy[0])
heatmap_cbar_ax = fig_xy.add_subplot(top_grid[0, 2])
xy_histogram_max = max(
    float(np.max(np.histogram2d(samples[:, 0], samples[:, 1], bins=[x_edges, y_edges])[0]))
    for samples in (xy_h0, xy_h1)
)
xy_heatmap_norm = mpl.colors.LogNorm(vmin=1.0, vmax=xy_histogram_max)
xy_heatmap_cmap = mpl.colormaps["Blues"].copy()
xy_heatmap_cmap.set_bad("white")
xy_heatmap_cmap.set_under("white")
mesh = None
for panel_index, (ax, samples, title) in enumerate([
    (axes_xy[0], xy_h0, r"$H_0$"),
    (axes_xy[1], xy_h1, rf"$H_1$, $\Phi_R={CONVENTIONAL_HIST_PHASE:.2f}$"),
]):
    mesh = ax.hist2d(
        samples[:, 0],
        samples[:, 1],
        bins=[x_edges, y_edges],
        cmap=xy_heatmap_cmap,
        norm=xy_heatmap_norm,
        cmin=1,
    )
    ax.scatter([xy_center[0]], [xy_center[1]], s=20, color="black", marker="+", label=r"$E_0$")
    ax.set_title(title, fontsize=18)
    ax.set_xlabel(r"$K_x$", fontsize=17)
    if panel_index == 0:
        ax.set_ylabel(r"$K_y$", fontsize=17)
    else:
        ax.tick_params(axis="y", labelleft=False)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.15, linewidth=0.5)
xy_colorbar = fig_xy.colorbar(mesh[3], cax=heatmap_cbar_ax)
xy_colorbar.set_label("pseudo-experiments / bin", fontsize=17)
xy_colorbar.ax.tick_params(labelsize=14)

# In standardized centered coordinates, the polar direction of the H1 shift
# is the locally optimal transverse readout azimuth (modulo pi).
h1_ax = axes_xy[1]
h1_ax.plot(
    noncentrality_x,
    noncentrality_y,
    color="0.08",
    linestyle="--",
    linewidth=1.5,
    zorder=5,
)
phi_demo = math.radians(50.0)
signal_radius = noncentrality_radius
baseline_radius = 0.72 * signal_radius
arc_radius = 0.53 * signal_radius
sigma_x, sigma_y = np.sqrt(xy_variance)
arrow_tip = (
    xy_center[0] + sigma_x * signal_radius * math.cos(phi_demo),
    xy_center[1] + sigma_y * signal_radius * math.sin(phi_demo),
)
h1_ax.plot(
    [xy_center[0], xy_center[0] + sigma_x * baseline_radius],
    [xy_center[1], xy_center[1]],
    color="0.15",
    linewidth=1.0,
    zorder=5,
)
h1_ax.annotate(
    "",
    xy=arrow_tip,
    xytext=tuple(xy_center),
    arrowprops={"arrowstyle": "-|>", "color": "0.15", "lw": 1.35, "mutation_scale": 10},
    zorder=6,
)
arc_angles = np.linspace(0.0, phi_demo, 80)
h1_ax.plot(
    xy_center[0] + sigma_x * arc_radius * np.cos(arc_angles),
    xy_center[1] + sigma_y * arc_radius * np.sin(arc_angles),
    color="0.15",
    linewidth=1.15,
    zorder=6,
)
label_angle = 0.50 * phi_demo
label_radius = 0.76 * signal_radius
h1_ax.text(
    xy_center[0] + sigma_x * label_radius * math.cos(label_angle),
    xy_center[1] + sigma_y * label_radius * math.sin(label_angle),
    r"$\phi$",
    fontsize=15.0,
    ha="center",
    va="center",
    zorder=7,
)
q_lo, q_hi = np.percentile(np.concatenate([q_h0, q_h1]), [0.2, 99.8])
q_hi = float(q_hi)
q_bins = np.linspace(max(0.0, float(q_lo)), q_hi, CONVENTIONAL_BIN_COUNT + 1)
axes_xy[2].hist(q_h0, bins=q_bins, histtype="stepfilled", alpha=0.22, color="#2F5D7C")
axes_xy[2].hist(q_h0, bins=q_bins, histtype="step", linewidth=1.35, color="#2F5D7C", label=r"$H_0$")
axes_xy[2].hist(q_h1, bins=q_bins, histtype="stepfilled", alpha=0.20, color="#C2702C")
axes_xy[2].hist(q_h1, bins=q_bins, histtype="step", linewidth=1.35, color="#C2702C", label=r"$H_1$")
axes_xy[2].axvline(
    q_threshold,
    color="black",
    linestyle="--",
    linewidth=1.4,
    label=rf"$Q_\alpha$ boundary ($\alpha={ALPHA:.2f}$, $\gamma={q_boundary_reject_probability:.3f}$)",
)
axes_xy[2].set_xlabel(r"$Q_{xy}$", fontsize=17)
axes_xy[2].set_ylabel("pseudo-experiments / bin", fontsize=17)
axes_xy[2].tick_params(axis="both", labelsize=14)
axes_xy[2].grid(alpha=0.18, linewidth=0.6)
axes_xy[2].legend(frameon=False, fontsize=14, loc="upper right")

if SAVE_FIGURES:
    xy_pdf_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{CONVENTIONAL_PRESET}_conventional_xy_quadrature.pdf"
    xy_summary_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{CONVENTIONAL_PRESET}_conventional_xy_quadrature_summary.csv"
    fig_xy.savefig(xy_pdf_path)
    xy_quadrature_summary.to_csv(xy_summary_path, index=False)
    print(f"saved {xy_pdf_path}")
    print(f"saved {xy_summary_path}")

plt.show()


In [ ]:
summary_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_summary.csv"
setup_path = FIGURE_DIR / f"{OUTPUT_BASENAME}_{RUN_PRESET}_setup.json"
summary.to_csv(summary_path, index=False)
setup_to_save = setup
setup_path.write_text(json.dumps(setup_to_save, indent=2, sort_keys=True, default=str) + "\n")
print(f"saved {summary_path}")
print(f"saved {setup_path}")
